In [17]:
import geopandas as gpd


In [18]:
import geopandas as gpd
import pandas as pd
import pyogrio
from pathlib import Path

kml_dir = Path("data/MoD Zones/_extracted_kml")
kml_files = sorted(kml_dir.glob("*.kml"))

all_gdfs = []

for kml_path in kml_files:
    print("\nFILE:", kml_path.name)

    layers = pyogrio.list_layers(kml_path)
    print(layers)

    # layers might be a DataFrame OR a numpy array depending on version
    if hasattr(layers, "iterrows"):   # pandas DataFrame
        layer_names = layers["name"].tolist()
    else:  # numpy array like [[name, geomtype], ...]
        layer_names = [row[0] for row in layers]

    for layer_name in layer_names:
        gdf = gpd.read_file(kml_path, driver="KML", layer=layer_name)
        gdf["source_file"] = kml_path.name
        gdf["layer_name"] = layer_name
        all_gdfs.append(gdf)

zones = gpd.GeoDataFrame(pd.concat(all_gdfs, ignore_index=True))

print("\nTOTAL ROWS:", len(zones))
print("GEOM TYPES:\n", zones.geometry.geom_type.value_counts())
zones.head()



FILE: AP.kml
[['Anantapur' 'MultiPolygon Z']
 ['Identified Wind Potential Blocks' 'MultiPolygon Z']]

FILE: GJ.kml
[['shoreline25km buffer' 'MultiPolygon Z']
 ['GJ_Blocks_Intersect2' 'MultiPolygon Z']
 ['NOC not required from MoD(Subject to the conditions - Refer Attached Pdf provided by MoD)'
  'MultiPolygon Z']
 ['Jamnagar-new' 'MultiPolygon Z']
 ['Jamnagar_and_khambaliya_airfield_56km-new' 'MultiPolygon Z']
 ['NOC to be obtained' 'MultiPolygon Z']
 ['kutch_2_new' 'MultiPolygon Z']
 ['IAF' 'MultiPolygon Z']
 ['Amreli_2' 'MultiPolygon Z']
 ['No wtg mod remarks-14062024' 'MultiPolygon Z']
 ['No WTG Zone' 'MultiPolygon Z']
 ['JamANDkhamairfiled_20km-NEW' 'MultiPolygon Z']
 ['Jamnagar_2' 'MultiPolygon Z']
 ['Amreli-1new' 'MultiPolygon Z']
 ['Jamnagar_1' 'MultiPolygon Z']
 ['Rajkot' 'MultiPolygon Z']
 ['Kutch_1' 'MultiPolygon Z']
 ['Kutch_6NM' 'MultiPolygon Z']
 ['Gujarat' 'MultiPolygon Z']
 ['GJ_Blocks' 'MultiPolygon Z']]

FILE: KA.kml
[['Block 7 Karnataka' 'Unknown']
 ['Tumkur' 'MultiP

,Name,Description,geometry,source_file,layer_name
0,0,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...","MULTIPOLYGON Z (((77 13.75 0, 77.2 13.75 0, 77...",AP.kml,Anantapur
1,Kurnool,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...","MULTIPOLYGON Z (((77.15396 15.93156 0, 77.1608...",AP.kml,Identified Wind Potential Blocks
2,Cuddapah,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...","MULTIPOLYGON Z (((77.73905 15.62621 0, 77.7303...",AP.kml,Identified Wind Potential Blocks
3,Anantapur,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...","MULTIPOLYGON Z (((76.97312 15.16865 0, 76.9615...",AP.kml,Identified Wind Potential Blocks
4,Chittoor,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...","MULTIPOLYGON Z (((77.70778 14.41704 0, 77.6835...",AP.kml,Identified Wind Potential Blocks


In [19]:
zones.groupby(["source_file", "layer_name"])["geometry"] \
     .apply(lambda s: s.geom_type.value_counts().to_dict()) \
     .sort_values() \
     .tail(30)


source_file  layer_name                                        
MH.kml       puneandNDAAirfeld_20km                Point          NaN
                                                   Polygon        NaN
MP.kml       MP_bloacks                            Point          NaN
                                                   Polygon        NaN
RJ.kml       Identified Wind Potential Blocks-new  MultiPolygon   NaN
                                                   Point          NaN
             Jailsalmer_1                          Point          NaN
                                                   Polygon        NaN
             Jailsalmer_2                          Point          NaN
                                                   Polygon        NaN
             Jaisalmer airfield_20km               Point          NaN
                                                   Polygon        NaN
             NOC to be obtained                    MultiPolygon   NaN
                          

In [20]:
zones_poly = zones[zones.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
print(len(zones_poly), "polygon features")


115 polygon features


In [21]:
import pandas as pd
import re

def norm(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def classify_row(layer_name, name, desc):
    text = " ".join([norm(layer_name), norm(name), norm(desc)])

    # HIGH PRIORITY rules first (override everything)
    if "no wtg" in text or "no wind turbine" in text:
        return "no_wtg_zone"

    if "noc not required" in text:
        return "noc_not_required"
    if "noc to be obtained" in text:
        return "noc_required"

    # Wind potential blocks
    if "identified wind potential blocks" in text:
        return "wind_potential_block"

    # Airfield/distance buffers
    if "airfield" in text:
        return "airfield_buffer"
    if re.search(r"\b\d+\s*km\b", text) or "10nm" in text or "6nm" in text:
        return "distance_buffer"
    if "buffer" in text or "shoreline" in text:
        return "buffer"

    # Blocks
    if "block" in text or "blocks" in text:
        return "block"

    # Military
    if "iaf" in text:
        return "military_zone"

    # State grouping
    ln = norm(layer_name)
    if ln in {"mh", "gujarat"}:
        return "state_zone"

    # Default: district/region
    return "district_zone"


zones_poly["zone_category"] = zones_poly.apply(
    lambda r: classify_row(r.get("layer_name"), r.get("Name"), r.get("Description")),
    axis=1
)

zones_poly["zone_category"].value_counts()


zone_category
wind_potential_block    31
district_zone           26
airfield_buffer         22
block                   18
no_wtg_zone              5
noc_required             5
noc_not_required         3
distance_buffer          3
military_zone            1
state_zone               1
Name: count, dtype: int64

In [22]:
zones[zones["source_file"].isna() | (zones["source_file"] == "")][["Name","layer_name"]].head(20)


,Name,layer_name


In [23]:
zones_poly = zones_poly.set_crs(epsg=4326, allow_override=True)

# fix invalid polygons
zones_poly["geometry"] = zones_poly["geometry"].buffer(0)

# drop empty/invalid
zones_poly = zones_poly[~zones_poly.geometry.is_empty]
zones_poly = zones_poly[zones_poly.geometry.notnull()]


In [24]:
is_gujarat = zones_poly["source_file"].str.upper().eq("GJ.kml")


In [25]:
def is_block(row):
    text = " ".join([
        str(row.get("layer_name", "")),
        str(row.get("Name", "")),
        str(row.get("Description", ""))
    ]).lower()

    return "block" in text


In [29]:
def split_gujarat_blocks(row):
    # Only touch Gujarat blocks
    if row["zone_category"] != "block":
        return row["zone_category"]

    if row["source_file"].upper() != "GJ.KML":
        return row["zone_category"]

    block_name = str(row.get("layer_name", "")).strip().lower()

    # normalize: spaces → _, remove weird chars
    block_name = (
        block_name
        .replace(" ", "_")
        .replace("-", "_")
    )

    return f"gj_block_{block_name}"


zones_poly["zone_category"] = zones_poly.apply(split_gujarat_blocks, axis=1)
zones_poly["zone_category"].value_counts()


zone_category
wind_potential_block             31
district_zone                    26
airfield_buffer                  22
gj_block_jamnagar_new             6
gj_block_gj_blocks                6
block                             5
no_wtg_zone                       5
noc_required                      5
distance_buffer                   3
noc_not_required                  3
gj_block_gj_blocks_intersect2     1
military_zone                     1
state_zone                        1
Name: count, dtype: int64

In [27]:
gj_blocks = zones_poly[zones_poly["zone_category"].str.startswith("gj_block")]

gj_blocks.groupby("zone_category").size()


Series([], dtype: int64)

In [31]:
print(zones_poly.columns)

Index(['Name', 'Description', 'geometry', 'source_file', 'layer_name',
       'zone_category'],
      dtype='object')


In [32]:
zones_poly["Description"].dropna().head(5)

0    <html xmlns:fo="http://www.w3.org/1999/XSL/For...
1    <html xmlns:fo="http://www.w3.org/1999/XSL/For...
2    <html xmlns:fo="http://www.w3.org/1999/XSL/For...
3    <html xmlns:fo="http://www.w3.org/1999/XSL/For...
4    <html xmlns:fo="http://www.w3.org/1999/XSL/For...
Name: Description, dtype: object

In [33]:
from lxml import etree

kml_path = "data/MoD Zones/_extracted_kml/GJ.kml"

tree = etree.parse(kml_path)
root = tree.getroot()

ns = {"kml": "http://www.opengis.net/kml/2.2"}


In [34]:
styles = {}

for style in root.findall(".//kml:Style", ns):
    style_id = style.get("id")

    poly = style.find(".//kml:PolyStyle/kml:color", ns)
    line = style.find(".//kml:LineStyle/kml:color", ns)

    styles[style_id] = {
        "poly_color": poly.text if poly is not None else None,
        "line_color": line.text if line is not None else None,
    }

styles


{'PolyStyle00': {'poly_color': 'ff73ffff', 'line_color': 'ffc5ff00'},
 'PolyStyle10': {'poly_color': 'ff00a838', 'line_color': 'ffff7000'},
 'PolyStyle20': {'poly_color': 'ff00a838', 'line_color': 'ffe65c00'},
 'PolyStyle40': {'poly_color': 'ffbeffff', 'line_color': 'ffc5ff00'},
 'PolyStyle80': {'poly_color': 'ff00ffff', 'line_color': 'ffc5ff00'},
 'PolyStyle110': {'poly_color': 'ff0000ff', 'line_color': 'ff000000'},
 'PolyStyle200': {'poly_color': '00f0f0f0', 'line_color': 'ff000000'}}

In [35]:
placemark_styles = []

for pm in root.findall(".//kml:Placemark", ns):
    name = pm.findtext("kml:name", default="", namespaces=ns)
    style_url = pm.findtext("kml:styleUrl", default="", namespaces=ns)

    placemark_styles.append({
        "Name": name,
        "styleUrl": style_url.replace("#", "")
    })

placemark_styles[:5]


[{'Name': 'Shoreline-Block-4', 'styleUrl': 'PolyStyle00'},
 {'Name': 'Kutch', 'styleUrl': 'PolyStyle10'},
 {'Name': 'Rajkot', 'styleUrl': 'PolyStyle20'},
 {'Name': 'Amreli', 'styleUrl': 'PolyStyle20'},
 {'Name': 'Surat', 'styleUrl': 'PolyStyle20'}]

In [36]:
import pandas as pd

style_df = pd.DataFrame(placemark_styles)
style_df = style_df.merge(
    pd.DataFrame.from_dict(styles, orient="index").reset_index(),
    left_on="styleUrl",
    right_on="index",
    how="left"
)

zones_poly = zones_poly.merge(style_df, on="Name", how="left")


In [38]:
import pandas as pd

def kml_to_hex(kml_color):
    if pd.isna(kml_color):
        return None

    kml_color = str(kml_color).strip()

    if len(kml_color) != 8:
        return None

    # KML format: aabbggrr
    rr = kml_color[6:8]
    gg = kml_color[4:6]
    bb = kml_color[2:4]

    return f"#{rr}{gg}{bb}"


zones_poly["fill_color"] = zones_poly["poly_color"].apply(kml_to_hex)
zones_poly["fill_color"].dropna().unique()


array(['#ffff00', '#ff0000', '#ffff73', '#38a800', '#ffffbe', '#f0f0f0'],
      dtype=object)

In [39]:
zones_poly["poly_color"].notna().sum()


np.int64(290)

In [50]:
import folium

center = [zones_poly.geometry.centroid.y.mean(), zones_poly.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=5, tiles="CartoDB positron")

def style_fn(feat):
    return {
        "fillColor": feat["properties"].get("fill_color", "#3388ff"),
        "color": "#000000",
        "weight": 1,
        "fillOpacity": 0.5,
    }

for cat, sub in zones_poly.groupby("zone_category"):
    folium.GeoJson(
        sub.to_json(),
        style_function=style_fn,
        name=f"{cat} ({len(sub)})",
        tooltip=folium.GeoJsonTooltip(fields=["Name", "layer_name", "zone_category", "Description"])
    ).add_to(m)




folium.LayerControl().add_to(m)
m.save("zones_preview.html")
print("Saved zones_preview.html — open it in your browser")

C:\Users\45579\AppData\Local\Temp\ipykernel_8068\2094908467.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [zones_poly.geometry.centroid.y.mean(), zones_poly.geometry.centroid.x.mean()]


Saved zones_preview.html — open it in your browser


In [15]:
gj_blocks = zones_poly[zones_poly["zone_category"].str.startswith("gj_block")]

gj_blocks.groupby("zone_category").size()


Series([], dtype: int64)

In [44]:
zones_poly[["zone_category", "fill_color"]].dropna().groupby(
    ["zone_category", "fill_color"]
).size()


zone_category                  fill_color
airfield_buffer                #ff0000        12
                               #ffffbe        60
block                          #38a800         3
distance_buffer                #ff0000         1
                               #ffff73         1
district_zone                  #ff0000       101
                               #ffff00        40
                               #ffffbe         9
gj_block_gj_blocks             #38a800         6
gj_block_gj_blocks_intersect2  #38a800         1
gj_block_jamnagar_new          #ffffbe        36
military_zone                  #ff0000         5
                               #ffff00         2
no_wtg_zone                    #ff0000         3
                               #ffffbe         2
noc_not_required               #38a800         3
noc_required                   #ff0000         2
                               #ffffbe         2
state_zone                     #f0f0f0         1
dtype: int64

In [48]:
zones_poly[zones_poly["source_file"] == "GJ.kml"]["Description"].dropna().to_frame().to_html("descriptions.html")

In [47]:
def extract_keywords(text):
    if not text:
        return {}
    text = text.lower()
    return {
        "no_wtg": "no wtg" in text,
        "noc_required": "noc" in text and "required" in text,
        "noc_not_required": "noc not required" in text,
    }

zones_poly = pd.concat(
    [
        zones_poly,
        zones_poly["description_text"].apply(lambda x: pd.Series(extract_keywords(x)))
    ],
    axis=1
)


In [46]:
from bs4 import BeautifulSoup
import pandas as pd

def clean_description(html):
    if pd.isna(html):
        return None
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator=" ", strip=True)

zones_poly["description_text"] = zones_poly["Description"].apply(clean_description)
zones_poly["description_text"].dropna().head(5)


0    0 FID 0 Id 0
1    0 FID 0 Id 0
2    0 FID 0 Id 0
3    0 FID 0 Id 0
4    0 FID 0 Id 0
Name: description_text, dtype: object

In [49]:
print(zones_poly["Description"].dropna().iloc[0])


<html xmlns:fo="http://www.w3.org/1999/XSL/Format" xmlns:msxsl="urn:schemas-microsoft-com:xslt"> <head> <META http-equiv="Content-Type" content="text/html"> <meta http-equiv="content-type" content="text/html; charset=UTF-8"> </head> <body style="margin:0px 0px 0px 0px;overflow:auto;background:#FFFFFF;"> <table style="font-family:Arial,Verdana,Times;font-size:12px;text-align:left;width:100%;border-collapse:collapse;padding:3px 3px 3px 3px"> <tr style="text-align:center;font-weight:bold;background:#9CBCE2"> <td>0</td> </tr> <tr> <td> <table style="font-family:Arial,Verdana,Times;font-size:12px;text-align:left;width:100%;border-spacing:0px; padding:3px 3px 3px 3px"> <tr> <td>FID</td> <td>0</td> </tr> <tr bgcolor="#D4E4F3"> <td>Id</td> <td>0</td> </tr> </table> </td> </tr> </table> </body> </html>


In [62]:
def assign_mh_category(row):
    cat = row["zone_category"]
    layer = str(row.get("layer_name", "")).lower()
    name = str(row.get("Name", "")).lower()
    desc = str(row.get("description_text", "")).lower()

    # 1️⃣ Special Limited Height (highest)
    if "block b" in name:
        return "SPECIAL_LIMITED_HEIGHT"

    # 2️⃣ Special Allowed (highest)
    if "block a" in name:
        return "SPECIAL_ALLOWED"

    # 3️⃣ NOC required
    if (
        "stara" in layer
        or "fid 2" in desc
        or ("buff_dist 56" in desc and cat == "airfield_buffer")
    ):
        return "NOC"

    # 4️⃣ NO_NOC
    if cat == "wind_potential_block":
        return "NO_NOC"

    # 5️⃣ NO_WTG (default)
    if (
        cat == "district_zone"
        or cat == "distance_buffer"
        or cat == "airfield_buffer"
    ):
        return "NO_WTG"

    return "NO_WTG"


In [63]:
mh_mask = zones_poly["source_file"].str.strip().str.lower() == "mh.kml"

mh_zones = zones_poly[mh_mask].copy()


In [64]:
mh_zones["mh_final_category"] = mh_zones.apply(assign_mh_category, axis=1)

mh_zones["mh_final_category"].value_counts()


mh_final_category
NO_WTG                    59
NOC                       17
NO_NOC                     4
SPECIAL_ALLOWED            1
SPECIAL_LIMITED_HEIGHT     1
Name: count, dtype: int64

In [55]:
zones_poly["mh_final_category"] = None  # initialize

zones_poly.loc[mh_mask, "mh_final_category"] = \
    mh_zones["mh_final_category"]


In [56]:
zones_poly["mh_final_category"] = None  # initialize

zones_poly.loc[mh_mask, "mh_final_category"] = \
    mh_zones["mh_final_category"]


In [57]:
mh_mask = zones_poly["source_file"].str.strip().str.lower() == "mh.kml"
mh_zones = zones_poly[mh_mask].copy()

print("Total MH zones:", len(mh_zones))
mh_zones["mh_final_category"].value_counts()


Total MH zones: 82


mh_final_category
NO_WTG    61
NOC       17
NO_NOC     4
Name: count, dtype: int64

In [65]:
centroids = mh_zones.to_crs(epsg=3857).geometry.centroid.to_crs(epsg=4326)
center = [centroids.y.mean(), centroids.x.mean()]


In [66]:
import folium

m = folium.Map(location=center, zoom_start=7, tiles="CartoDB positron")

CATEGORY_COLORS = {
    "NO_WTG": "#d73027",                 # red
    "NOC": "#fc8d59",                    # orange
    "NO_NOC": "#1a9850",                 # green
    "SPECIAL_ALLOWED": "#2c7bb6",        # blue
    "SPECIAL_LIMITED_HEIGHT": "#984ea3", # purple
}

for cat, sub in mh_zones.groupby("mh_final_category"):
    fg = folium.FeatureGroup(name=f"{cat} ({len(sub)})", show=False)

    def style_fn(feature, cat=cat):
        return {
            "fillColor": CATEGORY_COLORS.get(cat, "#cccccc"),
            "color": "#000000",
            "weight": 1,
            "fillOpacity": 0.6,
        }

    folium.GeoJson(
        sub.to_json(),
        style_function=style_fn,
        tooltip=folium.GeoJsonTooltip(
            fields=["layer_name", "zone_category", "mh_final_category"]
        ),
    ).add_to(fg)

    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("mh_zones_debug.html")
print("Saved mh_zones_debug.html — open it in browser")


Saved mh_zones_debug.html — open it in browser


In [67]:
gj_mask = zones_poly["source_file"].str.strip().str.lower() == "gj.kml"
gj_zones = zones_poly[gj_mask].copy()


In [79]:
def assign_gj_category(row):
    cat = row["zone_category"]
    name = row["Name"].lower()
    layer = str(row.get("layer_name", "")).lower()
    desc = str(row.get("description_text", "")).lower()

    # 1️⃣ shoreline buffer special case
    if "shoreline25km buffer" in layer:
        return "NOC"

    # 2️⃣ explicit NOC / NO_NOC categories
    if cat == "noc_required":
        return "NOC"

    if cat == "noc_not_required":
        return "NO_NOC"

    # 3️⃣ airfield buffer
    if cat == "airfield_buffer":
        if "56" in desc:
            return "NOC"
        return "NO_WTG"

    # 4️⃣ distance buffer
    if cat == "distance_buffer":
        return "NO_WTG"

    # 5️⃣ district zone
    if cat == "district_zone":
        if layer in ["amreli_2", "kutch_2_new"]:
            return "NOC"
        return "NO_WTG"
    
    if layer == "gj_blocks":
        if "block 4" in name:
            return "NO_NOC"

    # 6️⃣ Jamnagar new block
    if "jamnagar_new" in cat:
        return "NOC"
    
    if "intersect2" in cat:
        return "NO_NOC"

    # 7️⃣ military zone
    if cat == "military_zone":
        return "NO_WTG"

    # 8️⃣ no_wtg_zone
    if cat == "no_wtg_zone":
        return "NO_WTG"

    # Everything else ignored
    return None


In [80]:
gj_zones["gj_final_category"] = gj_zones.apply(assign_gj_category, axis=1)
gj_zones["gj_final_category"].value_counts()


gj_final_category
NOC       117
NO_WTG     61
NO_NOC      5
Name: count, dtype: int64

In [81]:
import folium

m = folium.Map(location=center, zoom_start=7, tiles="CartoDB positron")

CATEGORY_COLORS = {
    "NO_WTG": "#d73027",                 # red
    "NOC": "#fc8d59",                    # orange
    "NO_NOC": "#1a9850",                 # green
    "SPECIAL_ALLOWED": "#2c7bb6",        # blue
    "SPECIAL_LIMITED_HEIGHT": "#984ea3", # purple
}

for cat, sub in gj_zones.groupby("gj_final_category"):
    fg = folium.FeatureGroup(name=f"{cat} ({len(sub)})", show=False)

    def style_fn(feature, cat=cat):
        return {
            "fillColor": CATEGORY_COLORS.get(cat, "#cccccc"),
            "color": "#000000",
            "weight": 1,
            "fillOpacity": 0.6,
        }

    folium.GeoJson(
        sub.to_json(),
        style_function=style_fn,
        tooltip=folium.GeoJsonTooltip(
            fields=["layer_name", "zone_category", "gj_final_category"]
        ),
    ).add_to(fg)

    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("gj_zones_debug.html")
print("Saved gj_zones_debug.html — open it in browser")


Saved gj_zones_debug.html — open it in browser


In [83]:
rj_mask = zones_poly["source_file"].str.strip().str.lower() == "rj.kml"
rj_zones = zones_poly[rj_mask].copy()

print("Total RJ zones:", len(rj_zones))


Total RJ zones: 23


In [84]:
def assign_rj_category(row):
    cat = row["zone_category"]

    if cat == "noc_required":
        return "NOC"

    if cat == "wind_potential_block":
        return "NO_NOC"

    if cat in ["airfield_buffer", "district_zone"]:
        return "NO_WTG"

    return None  # ignore everything else


In [85]:
rj_zones["rj_final_category"] = rj_zones.apply(assign_rj_category, axis=1)
rj_zones["rj_final_category"].value_counts()


rj_final_category
NO_WTG    16
NO_NOC     4
NOC        3
Name: count, dtype: int64

In [86]:
rj_zones.groupby(["zone_category", "rj_final_category"]).size()


zone_category         rj_final_category
airfield_buffer       NO_WTG                2
district_zone         NO_WTG               14
noc_required          NOC                   3
wind_potential_block  NO_NOC                4
dtype: int64

In [87]:
import folium

m = folium.Map(location=center, zoom_start=7, tiles="CartoDB positron")

CATEGORY_COLORS = {
    "NO_WTG": "#d73027",                 # red
    "NOC": "#fc8d59",                    # orange
    "NO_NOC": "#1a9850",                 # green
    "SPECIAL_ALLOWED": "#2c7bb6",        # blue
    "SPECIAL_LIMITED_HEIGHT": "#984ea3", # purple
}

for cat, sub in rj_zones.groupby("rj_final_category"):
    fg = folium.FeatureGroup(name=f"{cat} ({len(sub)})", show=False)

    def style_fn(feature, cat=cat):
        return {
            "fillColor": CATEGORY_COLORS.get(cat, "#cccccc"),
            "color": "#000000",
            "weight": 1,
            "fillOpacity": 0.6,
        }

    folium.GeoJson(
        sub.to_json(),
        style_function=style_fn,
        tooltip=folium.GeoJsonTooltip(
            fields=["layer_name", "zone_category", "rj_final_category"]
        ),
    ).add_to(fg)

    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("rj_zones_debug.html")
print("Saved rj_zones_debug.html — open it in browser")

Saved rj_zones_debug.html — open it in browser


In [88]:
tn_mask = zones_poly["source_file"].str.strip().str.lower() == "tn.kml"
tn_zones = zones_poly[tn_mask].copy()

print("Total TN zones:", len(tn_zones))


Total TN zones: 15


In [89]:
import re

def assign_tn_category(row):
    cat = row["zone_category"]
    name = str(row.get("Name", "")).lower()
    desc = str(row.get("description_text", "")).lower()

    # 1️⃣ district_zone
    if cat == "district_zone":
        return "NO_WTG"

    # 2️⃣ airfield_buffer
    if cat == "airfield_buffer":
        if "56" in desc:
            return "NOC"
        return "NO_WTG"

    # 3️⃣ wind_potential_block
    if cat == "wind_potential_block":

        # Exception blocks (NOC required)
        if re.search(r"\bblock[\s\-_]?(9|7|6|2|4)\b", name + " " + desc):
            return "NOC"

        return "NO_NOC"

    return None


In [90]:
tn_zones["tn_final_category"] = tn_zones.apply(assign_tn_category, axis=1)
tn_zones["tn_final_category"].value_counts()


tn_final_category
NOC       7
NO_WTG    4
NO_NOC    4
Name: count, dtype: int64

In [91]:
import folium

m = folium.Map(location=center, zoom_start=7, tiles="CartoDB positron")

CATEGORY_COLORS = {
    "NO_WTG": "#d73027",                 # red
    "NOC": "#fc8d59",                    # orange
    "NO_NOC": "#1a9850",                 # green
    "SPECIAL_ALLOWED": "#2c7bb6",        # blue
    "SPECIAL_LIMITED_HEIGHT": "#984ea3", # purple
}

for cat, sub in tn_zones.groupby("tn_final_category"):
    fg = folium.FeatureGroup(name=f"{cat} ({len(sub)})", show=False)

    def style_fn(feature, cat=cat):
        return {
            "fillColor": CATEGORY_COLORS.get(cat, "#cccccc"),
            "color": "#000000",
            "weight": 1,
            "fillOpacity": 0.6,
        }

    folium.GeoJson(
        sub.to_json(),
        style_function=style_fn,
        tooltip=folium.GeoJsonTooltip(
            fields=["layer_name", "zone_category", "tn_final_category"]
        ),
    ).add_to(fg)

    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("tn_zones_debug.html")
print("Saved tn_zones_debug.html — open it in browser")

Saved tn_zones_debug.html — open it in browser


In [92]:
mp_mask = zones_poly["source_file"].str.strip().str.lower() == "mp.kml"
mp_zones = zones_poly[mp_mask].copy()

print("Total MP zones:", len(mp_zones))


Total MP zones: 3


In [93]:
def assign_mp_category(row):
    if row["zone_category"] == "block":
        return "NO_NOC"

    return None  # ignore everything else


In [94]:
mp_zones["mp_final_category"] = mp_zones.apply(assign_mp_category, axis=1)
mp_zones["mp_final_category"].value_counts()


mp_final_category
NO_NOC    3
Name: count, dtype: int64

In [95]:
import folium

m = folium.Map(location=center, zoom_start=7, tiles="CartoDB positron")

CATEGORY_COLORS = {
    "NO_WTG": "#d73027",                 # red
    "NOC": "#fc8d59",                    # orange
    "NO_NOC": "#1a9850",                 # green
    "SPECIAL_ALLOWED": "#2c7bb6",        # blue
    "SPECIAL_LIMITED_HEIGHT": "#984ea3", # purple
}

for cat, sub in mp_zones.groupby("mp_final_category"):
    fg = folium.FeatureGroup(name=f"{cat} ({len(sub)})", show=False)

    def style_fn(feature, cat=cat):
        return {
            "fillColor": CATEGORY_COLORS.get(cat, "#cccccc"),
            "color": "#000000",
            "weight": 1,
            "fillOpacity": 0.6,
        }

    folium.GeoJson(
        sub.to_json(),
        style_function=style_fn,
        tooltip=folium.GeoJsonTooltip(
            fields=["layer_name", "zone_category", "mp_final_category"]
        ),
    ).add_to(fg)

    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("mp_zones_debug.html")
print("Saved mp_zones_debug.html — open it in browser")

Saved mp_zones_debug.html — open it in browser


In [96]:
ap_mask = zones_poly["source_file"].str.strip().str.lower() == "ap.kml"
ap_zones = zones_poly[ap_mask].copy()

print("Total AP zones:", len(ap_zones))


Total AP zones: 11


In [97]:
def assign_ap_category(row):
    cat = row["zone_category"]

    if cat == "wind_potential_block":
        return "NO_NOC"

    if cat == "district_zone":
        return "NO_WTG"

    return None


In [98]:
ap_zones["ap_final_category"] = ap_zones.apply(assign_ap_category, axis=1)
ap_zones["ap_final_category"].value_counts()


ap_final_category
NO_WTG    7
NO_NOC    4
Name: count, dtype: int64

In [99]:
import folium

m = folium.Map(location=center, zoom_start=7, tiles="CartoDB positron")

CATEGORY_COLORS = {
    "NO_WTG": "#d73027",                 # red
    "NOC": "#fc8d59",                    # orange
    "NO_NOC": "#1a9850",                 # green
    "SPECIAL_ALLOWED": "#2c7bb6",        # blue
    "SPECIAL_LIMITED_HEIGHT": "#984ea3", # purple
}

for cat, sub in ap_zones.groupby("ap_final_category"):
    fg = folium.FeatureGroup(name=f"{cat} ({len(sub)})", show=False)

    def style_fn(feature, cat=cat):
        return {
            "fillColor": CATEGORY_COLORS.get(cat, "#cccccc"),
            "color": "#000000",
            "weight": 1,
            "fillOpacity": 0.6,
        }

    folium.GeoJson(
        sub.to_json(),
        style_function=style_fn,
        tooltip=folium.GeoJsonTooltip(
            fields=["layer_name", "zone_category", "ap_final_category"]
        ),
    ).add_to(fg)

    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("ap_zones_debug.html")
print("Saved ap_zones_debug.html — open it in browser")

Saved ap_zones_debug.html — open it in browser


In [100]:
ka_mask = zones_poly["source_file"].str.strip().str.lower() == "ka.kml"
ka_zones = zones_poly[ka_mask].copy()

print("Total KA zones:", len(ka_zones))

Total KA zones: 18


In [101]:
def assign_ka_category(row):
    cat = row["zone_category"]

    if cat in ["district_zone", "no_wtg_zone"]:
        return "NO_WTG"

    if cat == "wind_potential_block":
        return "NO_NOC"

    return None


In [102]:
ka_zones["ka_final_category"] = ka_zones.apply(assign_ka_category, axis=1)
ka_zones["ka_final_category"].value_counts()


ka_final_category
NO_WTG    9
NO_NOC    9
Name: count, dtype: int64

In [103]:
import folium

m = folium.Map(location=center, zoom_start=7, tiles="CartoDB positron")

CATEGORY_COLORS = {
    "NO_WTG": "#d73027",                 # red
    "NOC": "#fc8d59",                    # orange
    "NO_NOC": "#1a9850",                 # green
    "SPECIAL_ALLOWED": "#2c7bb6",        # blue
    "SPECIAL_LIMITED_HEIGHT": "#984ea3", # purple
}

for cat, sub in ka_zones.groupby("ka_final_category"):
    fg = folium.FeatureGroup(name=f"{cat} ({len(sub)})", show=False)

    def style_fn(feature, cat=cat):
        return {
            "fillColor": CATEGORY_COLORS.get(cat, "#cccccc"),
            "color": "#000000",
            "weight": 1,
            "fillOpacity": 0.6,
        }

    folium.GeoJson(
        sub.to_json(),
        style_function=style_fn,
        tooltip=folium.GeoJsonTooltip(
            fields=["layer_name", "zone_category", "ka_final_category"]
        ),
    ).add_to(fg)

    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("ka_zones_debug.html")
print("Saved ka_zones_debug.html — open it in browser")

Saved ka_zones_debug.html — open it in browser


In [104]:
zones_poly["final_category"] = None


In [107]:
zones_poly["final_category"] = None

zones_poly.loc[mh_zones.index, "final_category"] = mh_zones["mh_final_category"]
zones_poly.loc[gj_zones.index, "final_category"] = gj_zones["gj_final_category"]
zones_poly.loc[rj_zones.index, "final_category"] = rj_zones["rj_final_category"]
zones_poly.loc[tn_zones.index, "final_category"] = tn_zones["tn_final_category"]
zones_poly.loc[mp_zones.index, "final_category"] = mp_zones["mp_final_category"]
zones_poly.loc[ap_zones.index, "final_category"] = ap_zones["ap_final_category"]
zones_poly.loc[ka_zones.index, "final_category"] = ka_zones["ka_final_category"]


In [108]:
zones_poly["final_category"].value_counts()


final_category
NO_WTG                    156
NOC                       144
NO_NOC                     33
SPECIAL_ALLOWED             1
SPECIAL_LIMITED_HEIGHT      1
Name: count, dtype: int64

In [109]:
zones_poly.groupby("source_file")["final_category"].count()


source_file
AP.kml     11
GJ.kml    183
KA.kml     18
MH.kml     82
MP.kml      3
RJ.kml     23
TN.kml     15
Name: final_category, dtype: int64

In [110]:
final_zones = zones_poly[
    ["Name", "layer_name", "zone_category", "final_category", "geometry"]
].copy()

final_zones = final_zones[final_zones["final_category"].notna()]


In [111]:
import folium

m = folium.Map(location=center, zoom_start=7, tiles="CartoDB positron")

CATEGORY_COLORS = {
    "NO_WTG": "#d73027",                 # red
    "NOC": "#defc59",                    # orange
    "NO_NOC": "#1a9850",                 # green
    "SPECIAL_ALLOWED": "#ff04c9",        # blue
    "SPECIAL_LIMITED_HEIGHT": "#ff04c9", # purple
}

for cat, sub in ka_zones.groupby("ka_final_category"):
    fg = folium.FeatureGroup(name=f"{cat} ({len(sub)})", show=False)

    def style_fn(feature, cat=cat):
        return {
            "fillColor": CATEGORY_COLORS.get(cat, "#cccccc"),
            "color": "#000000",
            "weight": 1,
            "fillOpacity": 0.6,
        }

    folium.GeoJson(
        sub.to_json(),
        style_function=style_fn,
        tooltip=folium.GeoJsonTooltip(
            fields=["layer_name", "zone_category", "ka_final_category"]
        ),
    ).add_to(fg)

    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("ka_zones_debug.html")
print("Saved ka_zones_debug.html — open it in browser")

Saved ka_zones_debug.html — open it in browser


In [112]:
CATEGORY_COLORS = {
    "NO_WTG": "#d73027",                 # red
    "NOC": "#defc59",                    # orange
    "NO_NOC": "#1a9850",                 # green
    "SPECIAL_ALLOWED": "#ff04c9",        # blue
    "SPECIAL_LIMITED_HEIGHT": "#ff04c9", # purple
}


In [113]:
import folium

m = folium.Map(location=center, zoom_start=6, tiles="CartoDB positron")

def style_fn(feature):
    cat = feature["properties"]["final_category"]
    return {
        "fillColor": CATEGORY_COLORS.get(cat, "#cccccc"),
        "color": "#000000",
        "weight": 1,
        "fillOpacity": 0.6,
    }

folium.GeoJson(
    final_zones.to_json(),
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(
        fields=["Name", "layer_name", "zone_category", "final_category"]
    ),
    name="Final Regulatory Zones"
).add_to(m)

m.save("final_regulatory_map.html")
print("Saved final_regulatory_map.html")


Saved final_regulatory_map.html


In [114]:
print(final_zones)

            Name                        layer_name         zone_category  \
0              0                         Anantapur         district_zone   
1              0                         Anantapur         district_zone   
2              0                         Anantapur         district_zone   
3              0                         Anantapur         district_zone   
4              0                         Anantapur         district_zone   
..           ...                               ...                   ...   
336        Theni  Identified Wind Potential Blocks  wind_potential_block   
337  Tirunelveli  Identified Wind Potential Blocks  wind_potential_block   
338  Kanyakumari  Identified Wind Potential Blocks  wind_potential_block   
339       Ramnad  Identified Wind Potential Blocks  wind_potential_block   
340    Tuticorin  Identified Wind Potential Blocks  wind_potential_block   

    final_category                                           geometry  
0           NO_